# 🦷 Dental X-ray Cavity Detection
### CPU-Friendly Transfer Learning with MobileNetV2

---

**Goal:** Classify panoramic dental X-ray images into two classes:
- `0 = no_cavity`
- `1 = cavity`

**Approach:** Transfer Learning with MobileNetV2 — a lightweight pretrained CNN backbone trained on ImageNet, adapted for dental X-ray classification.

**Dataset:** [TUFTS Dental X-ray Dataset](https://tufts-cs-ml.github.io/tufts-dental-db/) — 980 training images + 20 test images, with bounding-box CSV annotations used to create binary labels.

---

### Connection to College Labs

| Lab | Concept | Used Here |
|-----|---------|----------|
| Lab 06 | Keras Sequential, `compile`, `fit`, `evaluate` | Model building and training loop |
| Lab 07 | CNN, Dropout, Data Augmentation | MobileNetV2 backbone + augmentation pipeline |
| Lab 08 | Transfer Learning, Freeze / Fine-Tune | Phase 1 (frozen) → Phase 2 (fine-tune last 10 layers) |

---
## Section 0 — Imports & Constants

In [ ]:
import csv
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

print("TensorFlow version:", tf.__version__)

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────────
IMG_SIZE         = (96, 96)   # small for CPU speed; MobileNetV2 accepts this
BATCH_SIZE       = 16         # low memory usage on CPU
EPOCHS_HEAD      = 5          # Phase 1: train head only
EPOCHS_FINETUNE  = 3          # Phase 2: fine-tune last 10 layers
SEED             = 42

DATASET_PATH     = Path("dataset")             # relative — works on any machine
MODEL_DIR        = Path("models")
OUTPUT_DIR       = Path("outputs")
BEST_MODEL_PATH  = MODEL_DIR / "best_model.keras"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Constants set.")
print(f"Image size : {IMG_SIZE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Dataset    : {DATASET_PATH.resolve()}")

---
## Section 1 — Dataset Loading

The TUFTS dataset has:
- `Radiographs/training_images/` and `Radiographs/testing_images/` — panoramic X-ray images
- `bboxes/trainBoundryBoxes.csv` and `bboxes/testBoundryBoxes.csv` — bounding-box annotations

**Labeling rule (classification, not detection):**
```
Image appears in bbox CSV  →  cavity   (1)
Image not in bbox CSV      →  no_cavity (0)
```

We do **not** use the bounding-box coordinates — only the image filenames.

In [ ]:
def read_positive_image_ids(csv_path: Path) -> set:
    """Return filenames of images that appear in the bbox CSV = cavity images."""
    ids = set()
    with csv_path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            image_id = row.get("imageID", "").strip()
            if image_id:
                ids.add(image_id)
    return ids


def collect_split(tufts_root: Path, split: str):
    """Return (paths_array, labels_array) for one split."""
    image_dir = tufts_root / "Radiographs" / ("training_images" if split == "train" else "testing_images")
    csv_path  = tufts_root / "bboxes" / ("trainBoundryBoxes.csv" if split == "train" else "testBoundryBoxes.csv")

    positive_ids = read_positive_image_ids(csv_path)
    image_paths  = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

    # Label rule: 1 if filename is in CSV, else 0
    labels = np.array([1 if p.name in positive_ids else 0 for p in image_paths], dtype=np.int32)
    paths  = np.array([str(p) for p in image_paths])

    n_cavity    = int(labels.sum())
    n_no_cavity = int((labels == 0).sum())
    print(f"{split.title():5s}: {len(labels)} images | {n_cavity} cavity | {n_no_cavity} no_cavity")
    return paths, labels


# Locate TUFTS root
tufts_root = None
for candidate in (DATASET_PATH, DATASET_PATH / "TUFTS"):
    if (candidate / "Radiographs" / "training_images").exists():
        tufts_root = candidate
        break

if tufts_root is None:
    raise FileNotFoundError(f"TUFTS dataset not found under {DATASET_PATH.resolve()}")

print(f"Dataset root: {tufts_root.resolve()}")
train_paths, train_labels = collect_split(tufts_root, "train")
test_paths,  test_labels  = collect_split(tufts_root, "test")

### Class Weights

The training set is **imbalanced** (645 no_cavity vs 335 cavity).  
Without correction the model tends to predict the majority class.  
Class weights give the minority class more importance during training.

Formula:
```
weight_i = total_samples / (2 × count_i)
```

In [ ]:
def balanced_class_weights(labels: np.ndarray):
    counts = np.bincount(labels.astype(int), minlength=2)
    if np.any(counts == 0):
        return None
    total = float(len(labels))
    return {0: total / (2.0 * counts[0]),
            1: total / (2.0 * counts[1])}


class_weight = balanced_class_weights(train_labels)
class_names  = ["no_cavity", "cavity"]

print(f"Class weights: {class_weight}")
print(" → cavity gets a higher weight because it is the minority class")

### TensorFlow Data Pipeline

Each image is:
1. Read from disk
2. Decoded as 3-channel RGB
3. Resized to 96×96
4. Normalized from `0..255` → `0..1`

Then the dataset is **batched → cached → prefetched**.  
`cache()` keeps processed batches in memory; `prefetch()` prepares the next batch while the model trains on the current one — both critical for CPU speed.

In [ ]:
def load_image_from_path(path: tf.Tensor, label: tf.Tensor):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0          # normalize to 0..1
    label = tf.reshape(tf.cast(label, tf.float32), (1,))
    return image, label


def make_tf_dataset(paths, labels, batch_size, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    autotune = tf.data.AUTOTUNE
    return (
        ds
        .map(load_image_from_path, num_parallel_calls=autotune)
        .batch(batch_size)
        .cache()       # keep processed batches in memory
        .prefetch(autotune)  # prepare next batch while model trains
    )


train_ds = make_tf_dataset(train_paths, train_labels, BATCH_SIZE, shuffle=True)
test_ds  = make_tf_dataset(test_paths,  test_labels,  BATCH_SIZE, shuffle=False)  # never shuffle test

print(f"Training batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Test batches     : {tf.data.experimental.cardinality(test_ds).numpy()}")

### Visualise Sample Images

In [ ]:
images_batch, labels_batch = next(iter(train_ds.take(1)))

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, img, lbl in zip(axes.flat, images_batch[:8], labels_batch[:8]):
    ax.imshow(img)
    ax.set_title(class_names[int(lbl.numpy())], fontsize=11)
    ax.axis("off")
fig.suptitle("Sample Training Images (after resize + normalize)", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_images.png", dpi=150)
plt.show()

---
## Section 2 — Data Augmentation

Augmentation creates modified versions of training images so the model generalises better.

We keep it **light** — these are medical images.  
Heavy transforms (large rotations, colour jitter) can distort diagnostically important features.

| Layer | Purpose |
|-------|---------|
| `RandomFlip("horizontal")` | Left/right variation |
| `RandomRotation(0.05)` | Small tilt only (±18°) |
| `RandomZoom(0.1)` | Small zoom variation |

**Key design choice:** augmentation is applied to the **training dataset pipeline only** — not inside the model.  
This means `model.evaluate()` and `model.predict()` never see random transformations.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
    ],
    name="light_augmentation",
)


def apply_training_augmentation(ds, augmentation):
    """Apply augmentation only to training batches — never to validation or test."""
    def augment(images, labels):
        return augmentation(images, training=True), labels
    return ds.map(augment).prefetch(tf.data.AUTOTUNE)


aug_train_ds = apply_training_augmentation(train_ds, data_augmentation)
print("Augmented training dataset ready.")

In [ ]:
# Show the same image 9 times with different random augmentations
images_batch, _ = next(iter(train_ds.take(1)))
sample_image = images_batch[0]

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax in axes.flat:
    aug = data_augmentation(tf.expand_dims(sample_image, 0), training=True)[0]
    ax.imshow(aug)
    ax.axis("off")
fig.suptitle("Same Image — 9 Random Augmentations", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "augmented_samples.png", dpi=150)
plt.show()

---
## Section 3 — Model Architecture (Transfer Learning)

### Why Transfer Learning?

Training a CNN from scratch needs a large dataset and strong hardware.  
The TUFTS test set has only **20 images** — far too small for scratch training.

Transfer learning reuses a CNN already trained on ImageNet (1.2M images).  
Even though ImageNet is not medical data, early CNN layers learn **general features** — edges, curves, textures — that are still useful for X-ray images.

### Why MobileNetV2?
- Lightweight — uses **depthwise separable convolutions** (~9× fewer operations than standard Conv2D)
- CPU-friendly — trains in minutes, not hours
- Available in Keras with pretrained ImageNet weights

### Model Structure
```
Input(96, 96, 3)
    ↓
MobileNetV2 backbone  ← frozen in Phase 1, last 10 layers unfrozen in Phase 2
    ↓
GlobalAveragePooling2D  ← converts feature maps to a compact vector
    ↓
Dense(64, relu)         ← learns task-specific patterns
    ↓
Dropout(0.3)            ← reduces overfitting
    ↓
Dense(1, sigmoid)       ← outputs P(cavity) between 0 and 1
```

**Why sigmoid?** Binary classification — one output neuron outputs probability of the positive class.  
Output ≥ 0.5 → cavity. Output < 0.5 → no_cavity.

In [ ]:
# Load MobileNetV2 backbone — pretrained on ImageNet, no top classifier
try:
    base_model = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3)
    )
except Exception as e:
    print(f"ImageNet weights unavailable ({e}). Using random weights.")
    base_model = tf.keras.applications.MobileNetV2(
        include_top=False, weights=None, input_shape=(*IMG_SIZE, 3)
    )

# Phase 1: freeze the entire backbone — only the new head will be trained
base_model.trainable = False

# Build the full classification model (Lab 06 Sequential pattern)
model = tf.keras.Sequential(
    [
        tf.keras.Input(shape=(*IMG_SIZE, 3)),
        base_model,                                       # pretrained feature extractor
        tf.keras.layers.GlobalAveragePooling2D(),         # flatten feature maps
        tf.keras.layers.Dense(64, activation="relu"),     # task-specific learning
        tf.keras.layers.Dropout(0.3),                     # regularisation
        tf.keras.layers.Dense(1, activation="sigmoid"),   # P(cavity)
    ],
    name="dental_model",
)

# Compile — binary crossentropy because this is a 2-class sigmoid output
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

trainable = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"\nTotal parameters    : {model.count_params():,}")
print(f"Trainable (Phase 1) : {trainable:,}  (head only — backbone frozen)")

---
## Section 4 — Phase 1 Training: Head Only (Backbone Frozen)

The entire MobileNetV2 backbone is **frozen** — its weights do not change.  
Only the two Dense layers (the new head) are trained.

**Why start this way?**
- Fast — only ~50K parameters updated instead of 2.2M
- Safe — cannot destroy useful pretrained features
- Gives the head a good starting point before we touch the backbone

**Callbacks:**
- `ModelCheckpoint` — saves the best model by validation accuracy
- `EarlyStopping(patience=2)` — stops if val_loss stops improving, saves CPU time

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2,
        restore_best_weights=True, verbose=1,
    ),
]
print("Callbacks ready.")

In [ ]:
print("Phase 1: training classification head — backbone frozen.")

history_phase1 = model.fit(
    aug_train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=2,             # one line per epoch — cleaner on CPU
)

print(f"\nPhase 1 final — train: {history_phase1.history['accuracy'][-1]*100:.2f}%  "
      f"val: {history_phase1.history['val_accuracy'][-1]*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, label in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(history_phase1.history[metric],          label=f"Train {label}")
    ax.plot(history_phase1.history[f"val_{metric}"], label=f"Val {label}")
    ax.set_title(label)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Phase 1: Head Only (Backbone Frozen)", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase1_history.png", dpi=150)
plt.show()

---
## Section 5 — Phase 2 Fine-Tuning: Unfreeze Last 10 Layers

After the head has learned something useful, we **unfreeze only the last 10 layers** of MobileNetV2 and let them adapt to our dental X-ray data.

**Why only the last 10 layers?**
- Early layers learn universal features (edges, textures) — no need to change these
- Later layers learn high-level, dataset-specific patterns — these benefit from fine-tuning
- Fewer trainable parameters = faster on CPU

**Why learning rate `1e-5` (100× smaller than Phase 1)?**  
Pretrained weights are already good. A large learning rate would destroy them.  
A very small rate makes careful, incremental updates — this is the key rule from **Lab 08 fine-tuning**.

In [ ]:
print(f"MobileNetV2 total layers: {len(base_model.layers)}")

# Unfreeze only the last 10 layers of the backbone
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

trainable_now = sum(l.trainable for l in base_model.layers)
print(f"Trainable backbone layers after unfreezing: {trainable_now} / {len(base_model.layers)}")

# Recompile with a much lower learning rate — critical for safe fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # 100x smaller than Phase 1
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("\nPhase 2: fine-tuning last 10 MobileNetV2 layers.")

history_phase2 = model.fit(
    aug_train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=2,
)

print(f"\nPhase 2 final — train: {history_phase2.history['accuracy'][-1]*100:.2f}%  "
      f"val: {history_phase2.history['val_accuracy'][-1]*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, label in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(history_phase2.history[metric],          label=f"Train {label}")
    ax.plot(history_phase2.history[f"val_{metric}"], label=f"Val {label}")
    ax.set_title(label)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Phase 2: Fine-Tuning Last 10 Layers", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase2_history.png", dpi=150)
plt.show()

---
## Section 6 — Combined Training History

Combine Phase 1 and Phase 2 into one continuous plot.  
The **red dashed line** marks where fine-tuning begins.

What to look for:
- Validation accuracy should improve or stay stable after the red line
- If validation accuracy drops after fine-tuning, the learning rate was too high

In [ ]:
# Combine Phase 1 + Phase 2 history into continuous curves
acc      = history_phase1.history["accuracy"]     + history_phase2.history["accuracy"]
val_acc  = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss     = history_phase1.history["loss"]         + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"]     + history_phase2.history["val_loss"]
p1_end   = len(history_phase1.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, train_vals, val_vals, label in zip(
    axes,
    [acc,  loss],
    [val_acc, val_loss],
    ["Accuracy", "Loss"],
):
    ax.plot(train_vals, label=f"Train {label}")
    ax.plot(val_vals,   label=f"Val {label}")
    ax.axvline(p1_end - 0.5, color="red", linestyle="--", label="Fine-tuning starts")
    ax.set_title(f"Model {label}")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Full Training History — Phase 1 + Phase 2", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_history.png", dpi=150)
plt.show()

---
## Section 7 — Evaluation

We load the **best saved model** (best validation accuracy across both phases) and evaluate on the test set.

Metrics reported:
- **Accuracy** — overall correct predictions
- **Precision** — of all predicted cavity, how many were truly cavity
- **Recall** — of all true cavity, how many did we catch
- **F1-score** — harmonic mean of precision and recall
- **Confusion matrix** — shows TP, TN, FP, FN visually

> **Note on the small test set:** The test set has only 20 images (4 cavity, 16 no_cavity).  
> One wrong prediction = 5% accuracy drop — so numbers can fluctuate.  
> A larger test set would give more reliable evaluation.

In [ ]:
# Load the best checkpoint saved during training
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

In [ ]:
# Collect all predictions from the test dataset
y_true, y_prob = [], []
for images, labels in test_ds:
    y_prob.extend(best_model.predict(images, verbose=0).reshape(-1))
    y_true.extend(labels.numpy().reshape(-1).astype(int))

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)  # threshold: >= 0.5 → cavity

print("Classification Report")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix
matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap="Blues")
ax.set_xticks([0, 1], labels=[f"Pred {n}" for n in class_names])
ax.set_yticks([0, 1], labels=[f"True {n}" for n in class_names])
ax.set_title("Confusion Matrix")
for r in range(2):
    for c in range(2):
        ax.text(c, r, str(matrix[r, c]), ha="center", va="center", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

print("\nMatrix layout:")
print("                 Pred no_cavity  |  Pred cavity")
print(f"True no_cavity       {matrix[0,0]} (TN)       |  {matrix[0,1]} (FP)")
print(f"True cavity          {matrix[1,0]} (FN)       |  {matrix[1,1]} (TP)")

---
## Section 8 — Prediction Visualisation

Show a 3×3 grid of test images with predicted and true labels.

- 🟢 **Green title** = correct prediction
- 🔴 **Red title** = wrong prediction
- **Conf** = model confidence for its predicted class

In [ ]:
images_batch, labels_batch = next(iter(test_ds.take(1)))
images_show = images_batch[:9]
labels_show = labels_batch[:9].numpy().reshape(-1).astype(int)
probs_show  = best_model.predict(images_show, verbose=0).reshape(-1)
preds_show  = (probs_show >= 0.5).astype(int)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax in axes.flat:
    ax.axis("off")

for ax, img, true, pred, prob in zip(axes.flat, images_show, labels_show, preds_show, probs_show):
    conf  = prob if pred == 1 else 1.0 - prob
    color = "green" if pred == true else "red"
    ax.imshow(img)
    ax.set_title(
        f"Pred: {class_names[pred]}\nTrue: {class_names[true]}\nConf: {conf*100:.1f}%",
        color=color, fontsize=10
    )
    ax.axis("off")

fig.suptitle("Model Predictions on Test X-rays  (green=correct, red=wrong)", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "predictions.png", dpi=150)
plt.show()

---
## Section 9 — Results Summary

In [ ]:
best_val    = max(val_acc)
total_epochs = len(acc)

print("=" * 52)
print("  DENTAL X-RAY DETECTION — RESULTS SUMMARY")
print("=" * 52)
print("  Model        : MobileNetV2 (Transfer Learning)")
print("  Image Size   : 96×96 px  (CPU optimized)")
print(f"  Total Epochs : {total_epochs}  (Phase 1: {EPOCHS_HEAD} + Phase 2: {EPOCHS_FINETUNE})")
print(f"  Best Val Acc : {best_val*100:.2f}%")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print("=" * 52)
print(f"\nSaved model  : {BEST_MODEL_PATH.resolve()}")
print(f"Saved figures: {OUTPUT_DIR.resolve()}")

---
## Conclusion

This project built a complete CPU-friendly dental X-ray cavity detection pipeline:

```
TUFTS images + bbox CSV
    → binary labels (cavity / no_cavity)
    → TensorFlow pipeline (resize 96×96, normalize, cache, prefetch)
    → light data augmentation (training only)
    → MobileNetV2 transfer learning
    → Phase 1: train head, backbone frozen
    → Phase 2: fine-tune last 10 layers, lr=1e-5
    → evaluate: accuracy, F1, confusion matrix, prediction grid
```

**Main strengths:**
- Real dataset with real medical images
- Full Lab 06 / 07 / 08 concepts applied
- CPU-friendly throughout (96×96, batch 16, MobileNetV2, EarlyStopping)
- Augmentation correctly applied to training data only
- Class weights handle dataset imbalance

**Known limitation:**  
Binary classification only — bounding boxes are used for labelling, not for object detection.
The test set (20 images) is small, so accuracy numbers can shift with individual predictions.